In [86]:
import json
import os
import pandas as pd
import numpy as np

def save_json(data, path):
    with open(path, "w") as f:
        json.dump(data, f, indent=2)

def story_mask(location, num_stories):
    """Vectorized story selector."""
    stories = np.arange(1, num_stories + 1)

    if location == "all":
        return np.ones(num_stories, dtype=bool)

    if location == "roof":
        mask = np.zeros(num_stories, dtype=bool)
        mask[-1] = True
        return mask

    if "--" in location:
        start, end = map(int, location.split("--"))
        return (stories >= start) & (stories <= end)

    loc_vec = np.array(location.split(), dtype=int)
    return np.isin(stories, loc_vec)

# load Pelicun outputs
model_dir = '../../examples/RCSW_4story_pelicun/'
with open(os.path.join(model_dir, 'input.json')) as file:
    pelicun_inputs = json.load(file)  

# Pull basic model info from Pelicun Inputs
num_stories = int(pelicun_inputs['DL']['Asset']['NumberOfStories'])
# TODO: replacement cost is still "manual"
if 'Repair' in pelicun_inputs['DL']['Losses']:
    total_cost = float(pelicun_inputs['DL']['Losses']['Repair']['ReplacementCost']['Median'])
else:
    total_cost = float(pelicun_inputs['DL']['Losses']['BldgRepair']['ReplacementCost']['Median'])
plan_area = float(pelicun_inputs['DL']['Asset']['PlanArea'])

# pull components 
comps = pd.read_csv(os.path.join(model_dir, 'CMP_QNT.csv'))

# repair cost realizations
DV_summary = pd.read_csv(os.path.join(model_dir, 'DL_summary.csv'))

# damage realizations
damage = pd.read_csv(os.path.join(model_dir, 'DMG_sample.csv'))

# decision variables
if os.path.exists(os.path.join(model_dir, "DV_repair_sample.csv")):
    dvs = pd.read_csv(os.path.join(model_dir, "DV_repair_sample.csv"))
else:
    dvs = pd.read_csv(os.path.join(model_dir, "DV_bldg_repair_sample.csv"))

# ds attributes
# TODO: switch this to __file__
static_data_dir = os.path.join(os.getcwd(), 'data')

ds_attributes = pd.read_csv(
    os.path.join(static_data_dir, "damage_state_attribute_mapping.csv")
)

# general inputs
with open(os.path.join(model_dir, 'general_inputs.json')) as file:
    general_inputs = json.load(file)  


# Filter DMG columns
frag_cols = damage.columns[
    damage.columns.str.match(r"^[B-F]")
]
damage = damage[frag_cols]
DMG_ids = damage.columns.tolist()

# Filter DV columns
DV_time = dvs.loc[:, dvs.columns.str.startswith("TIME")]
DV_cost = dvs.loc[:, dvs.columns.str.startswith("COST")]




C:\Users\hpham09\AppData\Local\Temp\ipykernel_27864\1968923768.py:50: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215) have mixed types. Specify dtype option on import or set low_memory=False.
  damage = pd.read_csv(os.path.join(model_dir, 'DMG_sample.csv'))
C:\Users\hph

In [87]:
DMG_ids

['B.10.44.001-1-1-0',
 'B.10.44.001-1-1-1',
 'B.10.44.001-1-1-2',
 'B.10.44.001-1-1-3',
 'B.10.44.001-1-2-0',
 'B.10.44.001-1-2-1',
 'B.10.44.001-1-2-2',
 'B.10.44.001-1-2-3',
 'B.10.44.001-2-1-0',
 'B.10.44.001-2-1-1',
 'B.10.44.001-2-1-2',
 'B.10.44.001-2-1-3',
 'B.10.44.001-2-2-0',
 'B.10.44.001-2-2-1',
 'B.10.44.001-2-2-2',
 'B.10.44.001-2-2-3',
 'B.10.44.001-3-1-0',
 'B.10.44.001-3-1-1',
 'B.10.44.001-3-1-2',
 'B.10.44.001-3-1-3',
 'B.10.44.001-3-2-0',
 'B.10.44.001-3-2-1',
 'B.10.44.001-3-2-2',
 'B.10.44.001-3-2-3',
 'B.10.44.001-4-1-0',
 'B.10.44.001-4-1-1',
 'B.10.44.001-4-1-2',
 'B.10.44.001-4-1-3',
 'B.10.44.001-4-2-0',
 'B.10.44.001-4-2-1',
 'B.10.44.001-4-2-2',
 'B.10.44.001-4-2-3',
 'B.10.49.012-1-1-0',
 'B.10.49.012-1-1-1',
 'B.10.49.012-1-1-2',
 'B.10.49.012-1-2-0',
 'B.10.49.012-1-2-1',
 'B.10.49.012-1-2-2',
 'B.10.49.012-2-1-0',
 'B.10.49.012-2-1-1',
 'B.10.49.012-2-1-2',
 'B.10.49.012-2-2-0',
 'B.10.49.012-2-2-1',
 'B.10.49.012-2-2-2',
 'B.10.49.012-3-1-0',
 'B.10.49.

In [88]:
# building_model.json

# count number of stairs in the building
stair_mask = comps["ID"].str.contains("C.20.11", regex=False)
# Assumes number of vertical egress routes is the min number of stairs on any story. This is faulty logic and wont hold true for all comp tables 
num_stairs = comps.loc[stair_mask, "Theta_0"].min() if stair_mask.any() else 0

# Count the number of elevator bays in the building
elev_mask = comps["ID"].str.contains("D.10.14", regex=False)
num_elev = comps.loc[elev_mask, "Theta_0"].max() if elev_mask.any() else 0

# construct building_model.json
building_model = dict(
    building_value=total_cost,
    num_stories=num_stories,
    area_per_story_sf=[plan_area] * num_stories,
    ht_per_story_ft=[general_inputs["typ_story_ht_ft"]] * num_stories,
    edge_lengths=[
        [general_inputs["length_side_1_ft"]] * num_stories,
        [general_inputs["length_side_2_ft"]] * num_stories,
    ],
    struct_bay_area_per_story=[
        general_inputs["typ_struct_bay_area_ft"]
    ] * num_stories,
    num_entry_doors=general_inputs["num_entry_doors"],
    num_elevators=int(num_elev),
    stairs_per_story=[int(num_stairs)] * num_stories,
    occupants_per_story=[
        general_inputs["peak_occ_rate"] * plan_area
    ] * num_stories,
)

save_json(building_model, os.path.join(model_dir, "building_model.json"))


In [89]:
DV_summary

,#,repair_cost-,repair_time-parallel,repair_time-sequential,collapse,irreparable
0,0,396789.269905,131.863271,321.008536,0.0,0.0
1,1,546627.119433,175.857869,449.451116,0.0,0.0
2,2,381045.718685,107.287372,258.618758,0.0,0.0
3,3,553380.530697,162.369157,394.059906,0.0,0.0
4,4,299568.874182,68.344962,174.598201,0.0,0.0
...,...,...,...,...,...,...
4995,4995,498483.903127,81.415173,300.381829,0.0,0.0
4996,4996,400783.228635,190.055320,381.858356,0.0,0.0
4997,4997,409013.370077,112.535418,307.607478,0.0,0.0
4998,4998,127774.782705,71.715381,144.840573,0.0,0.0


In [90]:
# construct damage_consequences.json

# determine replacement cases
replacement_mask = DV_summary["collapse"].astype('int') | DV_summary["irreparable"].astype('int')

# calculate repair cost ratio
damage_consequences = dict(
    repair_cost_ratio_total=(
        DV_summary["repair_cost-"] / total_cost
    ).tolist(),
    simulated_replacement_time=np.where(
        replacement_mask,
        DV_summary["repair_time-parallel"],
        np.nan,
    ).tolist(),
)

save_json(
    damage_consequences,
    os.path.join(model_dir, "damage_consequences.json"),
)


In [91]:
comps

,ID,Units,Location,Direction,Theta_0,Blocks,Family,Theta_1,Comment
0,B.20.22.001,SF,4,"1,2",4320.0,144,NaN,0.6,B2022.001 Curtain Walls - Generic Midrise S...
1,B.30.11.011,SF,4,0,3888.0,38,NaN,1.3,"B3011.011 Concrete tile roof, tiles secured..."
2,C.10.11.001a,LF,4,"1,2",1440.0,14,NaN,0.2,"C1011.001a Wall Partition, Type: Gypsum wit..."
3,C.30.11.001a,LF,4,"1,2",108.9,1,NaN,0.7,"C3011.001a Wall Partition, Type: Gypsum + W..."
4,C.30.27.001,SF,4,0,10800.0,108,NaN,0.2,"C3027.001 Raised Access Floor, non seismica..."
...,...,...,...,...,...,...,...,...,...
86,D.30.31.021a,EA,roof,0,3.0,3,NaN,0.1,D3031.021a Cooling Tower - Capacity: < 100 ...
87,D.30.52.011a,EA,roof,0,13.0,13,NaN,0.2,D3052.011a Air Handling Unit - Capacity: <5...
88,D.50.12.013a,EA,roof,0,2.9,2,NaN,0.5,D5012.013a Motor Control Center - Capacity:...
89,B.10.44.101,SF,1--4,"1,2",288.0,2,NaN,NaN,Slender concrete wall


In [92]:
# construct comp_population.csv

# list of stories and directions
stories = np.arange(1, num_stories + 1)
dirs = np.array([1, 2, 3])

# handle multi index
multi_index = pd.MultiIndex.from_product(
    [stories, dirs],
    names=["story", "dir"]
)

comp_population = pd.DataFrame(index=multi_index)

# build comp_population.csv
for _, comp in comps.iterrows():

    # remove first two periods 
    temp_string = comp["ID"].replace(".", "", 1)
    frag_id = temp_string.replace(".", "", 1)
    frag_id = frag_id.replace('.', '_')
    comp_population[frag_id] = 0.0

    story_sel = story_mask(comp["Location"], num_stories)

    dir_vec = np.array(
        str(comp["Direction"]).replace("0", "3").split(","),
        dtype=int
    )

    mask = (
        np.isin(comp_population.index.get_level_values("story"), stories[story_sel])
        & np.isin(comp_population.index.get_level_values("dir"), dir_vec)
    )

    comp_population.loc[mask, frag_id] += float(comp["Theta_0"])

comp_population.reset_index().to_csv(
    os.path.join(model_dir, "comp_population.csv"),
    index=False
)


In [93]:
comp_population

B2022_001  B3011_011  C1011_001a  C3011_001a  C3027_001  \
story dir                                                            
1     1       4320.0        0.0      1440.0       108.9        0.0   
      2       4320.0        0.0      1440.0       108.9        0.0   
      3          0.0     3888.0         0.0         0.0    10800.0   
2     1          0.0        0.0         0.0         0.0        0.0   
      2          0.0        0.0         0.0         0.0        0.0   
      3          0.0        0.0         0.0         0.0        0.0   
3     1          0.0        0.0         0.0         0.0        0.0   
      2          0.0        0.0         0.0         0.0        0.0   
      3          0.0        0.0         0.0         0.0        0.0   
4     1          0.0        0.0         0.0         0.0        0.0   
      2          0.0        0.0         0.0         0.0        0.0   
      3          0.0        0.0         0.0         0.0        0.0   

           C3032_001a  D2021_011a  D3041_012a  D3041_011a  D3041_031a  ...  \
story dir                                                              ...   
1     1           0.0         0.0         0.0         0.0         0.0  ...   
      2           0.0         0.0         0.0         0.0         0.0  ...   
      3         288.0       604.8       288.0      1080.0       129.6  ...   
2     1           0.0         0.0         0.0         0.0         0.0  ...   
      2           0.0         0.0         0.0         0.0         0.0  ...   
      3           0.0         0.0         0.0         0.0         0.0  ...   
3     1           0.0         0.0         0.0         0.0         0.0  ...   
      2           0.0         0.0         0.0         0.0         0.0  ...   
      3           0.0         0.0         0.0         0.0         0.0  ...   
4     1           0.0         0.0         0.0         0.0         0.0  ...   
      2           0.0         0.0         0.0         0.0         0.0  ...   
      3           0.0         0.0         0.0         0.0         0.0  ...   

           D4011_031a  C2011_001a  D5012_021a  D1014_011  D3031_011a  \
story dir                                                              
1     1           0.0         2.9         0.0        0.0         0.0   
      2           0.0         2.9         0.0        0.0         0.0   
      3         129.6         0.0         1.0        0.0         0.0   
2     1           0.0         0.0         0.0        0.0         0.0   
      2           0.0         0.0         0.0        0.0         0.0   
      3           0.0         0.0         0.0        0.0         0.0   
3     1           0.0         0.0         0.0        0.0         0.0   
      2           0.0         0.0         0.0        0.0         0.0   
      3           0.0         0.0         0.0        0.0         0.0   
4     1           0.0         0.0         0.0        0.0         0.0   
      2           0.0         0.0         0.0        0.0         0.0   
      3           0.0         0.0         0.0        2.0         3.0   

           D3031_021a  D3052_011a  D5012_013a  B1044_101  B1049_012  
story dir                                                            
1     1           0.0         0.0         0.0      288.0       21.0  
      2           0.0         0.0         0.0      288.0       21.0  
      3           0.0         0.0         0.0        0.0        0.0  
2     1           0.0         0.0         0.0      288.0       21.0  
      2           0.0         0.0         0.0      288.0       21.0  
      3           0.0         0.0         0.0        0.0        0.0  
3     1           0.0         0.0         0.0      288.0       21.0  
      2           0.0         0.0         0.0      288.0       21.0  
      3           0.0         0.0         0.0        0.0        0.0  
4     1           0.0         0.0         0.0      288.0       21.0  
      2           0.0         0.0         0.0      288.0       21.0  
      3           

In [ ]:
dmg_meta = []

for col in damage.columns:
    parts = col.split("-")

    dmg_meta.append({
        "column": col,
        "frag_id": f"{parts[0]}{parts[1]}{parts[2]}.{parts[3]}",
        "loc": int(parts[-3]),
        "dir": int(parts[-2]) or 3,
        "ds": int(parts[-1]),
    })

dmg_meta = pd.DataFrame(dmg_meta)

num_reals = len(damage)
num_ds = len(ds_attributes)

sim_damage = {
    s: {
        "qnt_damaged": np.zeros((num_reals, num_ds)),
        "worker_days": np.zeros((num_reals, num_ds)),
        "repair_cost": np.zeros((num_reals, num_ds)),
    }
    for s in range(1, num_stories + 1)
}

for _, meta in dmg_meta.iterrows():
    s = meta["loc"]
    col = meta["column"]

    if s not in sim_damage:
        continue

    dmg_data = damage[col].fillna(0).values

    sim_damage[s]["qnt_damaged"] += dmg_data[:, None]



In [96]:
dmg_meta

,column,frag_id,loc,dir,ds
0,B.10.44.001-1-1-0,B.10.44.00111.0,1,1,0
1,B.10.44.001-1-1-1,B.10.44.00111.1,1,1,1
2,B.10.44.001-1-1-2,B.10.44.00111.2,1,1,2
3,B.10.44.001-1-1-3,B.10.44.00111.3,1,1,3
4,B.10.44.001-1-2-0,B.10.44.00112.0,1,2,0
...,...,...,...,...,...
190,D.40.11.053a-3-0-1,D.40.11.053a30.1,3,3,1
191,D.40.11.053a-3-0-2,D.40.11.053a30.2,3,3,2
192,D.40.11.053a-4-0-0,D.40.11.053a40.0,4,3,0
193,D.40.11.053a-4-0-1,D.40.11.053a40.1,4,3,1


In [95]:
damage.columns

Index(['B.10.44.001-1-1-0', 'B.10.44.001-1-1-1', 'B.10.44.001-1-1-2',
       'B.10.44.001-1-1-3', 'B.10.44.001-1-2-0', 'B.10.44.001-1-2-1',
       'B.10.44.001-1-2-2', 'B.10.44.001-1-2-3', 'B.10.44.001-2-1-0',
       'B.10.44.001-2-1-1',
       ...
       'D.40.11.053a-1-0-2', 'D.40.11.053a-2-0-0', 'D.40.11.053a-2-0-1',
       'D.40.11.053a-2-0-2', 'D.40.11.053a-3-0-0', 'D.40.11.053a-3-0-1',
       'D.40.11.053a-3-0-2', 'D.40.11.053a-4-0-0', 'D.40.11.053a-4-0-1',
       'D.40.11.053a-4-0-2'],
      dtype='object', length=195)